# HealthConnect — Analyse initiale
**Filière :** Data Analytics — AnalystLab Africa



## 0. Chargement des données

In [1]:
import pandas as pd

df = pd.read_csv("HealthConnect_Appointment_Data.csv")
df["booking_date"] = pd.to_datetime(df["booking_date"])
df["appointment_date"] = pd.to_datetime(df["appointment_date"])

df.head()

,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2025-02-06,2025-02-18,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2026-02-25,2026-02-27,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,2025-11-16,2025-12-24,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,2025-07-18,2025-08-28,Thursday,Evening,41,3,1,Yes,SMS,7.4,35.0,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,2025-07-09,2025-08-25,Monday,Afternoon,47,3,1,Yes,Email,5.6,27.0,No-Show


## 1. Vue d'ensemble du dataset

On commence par la taille du fichier, les types de colonnes, et la période couverte.

In [2]:
print("Shape :", df.shape)
print()
print(df.dtypes)

Shape : (5000, 18)

appointment_id                      str
patient_id                          str
gender                              str
age                               int64
age_group                           str
appointment_type                    str
booking_date             datetime64[us]
appointment_date         datetime64[us]
appointment_day                     str
appointment_time                    str
booking_lead_days                 int64
previous_appointments             int64
previous_no_shows                 int64
reminder_sent                       str
reminder_channel                    str
distance_to_clinic_km           float64
waiting_time_minutes            float64
appointment_outcome                 str
dtype: object


In [3]:
print("Rendez-vous du", df["appointment_date"].min().date(), "au", df["appointment_date"].max().date())
print("Réservations du", df["booking_date"].min().date(), "au", df["booking_date"].max().date())

Rendez-vous du 2025-01-01 au 2026-06-30
Réservations du 2024-11-07 au 2026-06-27


In [4]:
n_patients = df["patient_id"].nunique()
print("Patients uniques :", n_patients, "/ Rendez-vous total :", len(df))
print("Moyenne de RDV par patient :", round(len(df) / n_patients, 2))
print("Max de RDV pour un même patient :", df["patient_id"].value_counts().max())

Patients uniques : 1696 / Rendez-vous total : 5000
Moyenne de RDV par patient : 2.95
Max de RDV pour un même patient : 9


## 2. Évaluation de la qualité des données

### 2.1 Valeurs manquantes

In [5]:
missing = df.isna().sum()
missing = missing[missing > 0]
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"nb_manquants": missing, "pct": missing_pct})

,nb_manquants,pct
reminder_channel,1366,27.32
distance_to_clinic_km,90,1.80
waiting_time_minutes,60,1.20


In [6]:
# reminder_channel a l'air d'avoir pas mal de manquants (~27%) : est-ce lié à reminder_sent ?
df.groupby("reminder_sent")["reminder_channel"].apply(lambda s: s.isna().sum())
# -> vide uniquement quand reminder_sent == "No", donc ce n'est pas une vraie donnée manquante

reminder_sent
No     1366
Yes       0
Name: reminder_channel, dtype: int64

### 2.2 Doublons

In [7]:
print("Lignes strictement dupliquées :", df.duplicated().sum())
print("appointment_id dupliqués :", df["appointment_id"].duplicated().sum())

Lignes strictement dupliquées : 0
appointment_id dupliqués : 0


### 2.3 Cohérence des dates

In [8]:
bad_order = (df["appointment_date"] < df["booking_date"]).sum()
print("RDV programmés avant leur réservation :", bad_order)

calc_lead = (df["appointment_date"] - df["booking_date"]).dt.days
mismatches = (calc_lead - df["booking_lead_days"]).abs()
print("Écarts entre booking_lead_days et le calcul réel :", (mismatches > 0).sum())

day_mismatch = (df["appointment_date"].dt.day_name() != df["appointment_day"]).sum()
print("Jours de semaine incohérents :", day_mismatch)

RDV programmés avant leur réservation : 0
Écarts entre booking_lead_days et le calcul réel : 0
Jours de semaine incohérents : 0


### 2.4 Valeurs aberrantes

In [9]:
print("age :", df["age"].min(), "-", df["age"].max())
print("booking_lead_days :", df["booking_lead_days"].min(), "-", df["booking_lead_days"].max())
print("distance_to_clinic_km :", df["distance_to_clinic_km"].min(), "-", df["distance_to_clinic_km"].max())
print("waiting_time_minutes :", df["waiting_time_minutes"].min(), "-", df["waiting_time_minutes"].max())


incoherent = (df["previous_no_shows"] > df["previous_appointments"]).sum()
print("previous_no_shows > previous_appointments :", incoherent)

age : 18 - 80
booking_lead_days : 0 - 60
distance_to_clinic_km : 0.5 - 45.0
waiting_time_minutes : 2.0 - 68.0
previous_no_shows > previous_appointments : 0


### 2.5 Cohérence des catégories

In [10]:
cat_cols = ["gender", "age_group", "appointment_type", "appointment_day",
            "appointment_time", "appointment_outcome", "reminder_sent"]

for c in cat_cols:
    print(c, "->", sorted(df[c].dropna().unique().tolist()))

gender -> ['Female', 'Male', 'Prefer not to say']
age_group -> ['18-24', '25-34', '35-44', '45-54', '55-64', '65+']
appointment_type -> ['Diagnostic Test', 'Follow-up', 'General Consultation', 'Specialist Consultation']
appointment_day -> ['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']
appointment_time -> ['Afternoon', 'Evening', 'Morning']
appointment_outcome -> ['Attended', 'Cancelled', 'No-Show']
reminder_sent -> ['No', 'Yes']


### 2.6 Équilibre de la variable cible

In [11]:
counts = df["appointment_outcome"].value_counts()
pct = df["appointment_outcome"].value_counts(normalize=True).mul(100).round(2)
pd.DataFrame({"nb": counts, "pct": pct})

,nb,pct
appointment_outcome,,
No-Show,2423,48.46
Attended,2314,46.28
Cancelled,263,5.26


**Conclusion :** dataset propre dans l'ensemble — pas de doublon, dates cohérentes entre elles, pas de valeur aberrante, catégories bien écrites. À traiter avant d'aller plus loin : recoder `reminder_channel` (catégorie "Aucun" plutôt qu'un NaN) et décider quoi faire des ~1-2% de manquants sur `distance_to_clinic_km` et `waiting_time_minutes` (imputation par la médiane, probablement).

## 3. Signaux préliminaires

Rien de définitif ici

In [12]:
noshow_rate = lambda s: (s == "No-Show").mean() * 100

print("Taux de no-show selon le rappel envoyé :")
print(df.groupby("reminder_sent")["appointment_outcome"].apply(noshow_rate).round(2))

Taux de no-show selon le rappel envoyé :
reminder_sent
No     51.39
Yes    47.36
Name: appointment_outcome, dtype: float64


In [13]:
df["had_prev_noshow"] = df["previous_no_shows"] > 0

print("Taux de no-show selon l'historique du patient :")
print(df.groupby("had_prev_noshow")["appointment_outcome"].apply(noshow_rate).round(2))

Taux de no-show selon l'historique du patient :
had_prev_noshow
False    43.51
True     55.41
Name: appointment_outcome, dtype: float64


Deux pistes qui se dégagent un peu : le taux de no-show est légèrement plus élevé sans rappel (51% contre 47%), et surtout nettement plus élevé chez les patients ayant déjà raté un rendez-vous (55% contre 43%). Ça confirme l'intérêt des questions business posées dans le document d'analyse — à creuser avec de vrais KPI la semaine prochaine.